In [5]:
#!pip install pdfminer
#!pip install PyPDF2
#!pip install pdfplumber

In [6]:
import pandas as pd
import os
import json
import numpy as np
from pdfminer.high_level import extract_text
from PyPDF2 import PdfReader
import pdfplumber

### Avaliação da raspagem

In [7]:
def verificar_relatorios_em_empresas(diretorio_base="pdfs"):
    print(f"\n Verificando relatórios na pasta base: {diretorio_base}\n")

    for empresa in os.listdir(diretorio_base):
        caminho_empresa = os.path.join(diretorio_base, empresa)
        if not os.path.isdir(caminho_empresa):
            continue

        print(f"\n Empresa: {empresa}")

        for pasta_ano in os.listdir(caminho_empresa):
            caminho_ano = os.path.join(caminho_empresa, pasta_ano)

            if os.path.isdir(caminho_ano):
                caminho_originais = os.path.join(caminho_ano, "originais")

                if os.path.exists(caminho_originais):
                    arquivos_pdf = [
                        f for f in os.listdir(caminho_originais)
                        if f.lower().endswith(".pdf")
                    ]
                    total = len(arquivos_pdf)

                    if total < 12:
                        print(f"{pasta_ano} - Apenas {total} relatórios encontrados (esperado: 12)")
                    else:
                        print(f"{pasta_ano} - OK ({total} relatórios)")
                else:
                    print(f"{pasta_ano} - Pasta 'originais' não encontrada")

verificar_relatorios_em_empresas("E:/Projetos/Github_ViniciusJ/Pesquisa_IC_IA/crawler-relacao-investidor/pdfs/Gerdau")


 Verificando relatórios na pasta base: E:/Projetos/Github_ViniciusJ/Pesquisa_IC_IA/crawler-relacao-investidor/pdfs/Gerdau


 Empresa: 2015
originais - Pasta 'originais' não encontrada
textos - Pasta 'originais' não encontrada
jsons - Pasta 'originais' não encontrada

 Empresa: 2016
originais - Pasta 'originais' não encontrada
textos - Pasta 'originais' não encontrada
jsons - Pasta 'originais' não encontrada

 Empresa: 2017
originais - Pasta 'originais' não encontrada
textos - Pasta 'originais' não encontrada
jsons - Pasta 'originais' não encontrada

 Empresa: 2018
originais - Pasta 'originais' não encontrada
textos - Pasta 'originais' não encontrada
jsons - Pasta 'originais' não encontrada

 Empresa: 2019
originais - Pasta 'originais' não encontrada
textos - Pasta 'originais' não encontrada
jsons - Pasta 'originais' não encontrada

 Empresa: 2020
originais - Pasta 'originais' não encontrada
textos - Pasta 'originais' não encontrada
jsons - Pasta 'originais' não encontrada

 Empresa: 2

### Avaliação da saida de texto no JSON

In [11]:
import os
import json
import numpy as np


def verificar_jsons_e_raw_text_em_empresas_problemas_apenas(diretorio_base="pdfs"):
    """
    Verifica arquivos JSON e o status de 'raw_text' dentro das estruturas
    de pastas de empresas e anos, retornando apenas informações sobre
    empresas/anos que possuem problemas.

    Problemas incluem:
    - Pasta 'jsons' não encontrada.
    - Nenhum arquivo JSON encontrado na pasta 'jsons'.
    - Arquivos JSON com 'raw_text' vazio ou ausente.
    - Arquivos JSON com 'raw_text' com menos de 100 caracteres.
    - Erros de leitura/parsing de arquivos JSON.

    Args:
        diretorio_base (str): O caminho para a pasta base que contém as pastas
                              das empresas (ex: 'pdfs').
    """
    print(f"\n--- Verificando arquivos JSON e 'raw_text' na pasta base: {diretorio_base} (Apenas Problemas) ---\n")

    # Percorre as pastas de cada empresa dentro do diretório base
    for nome_empresa in os.listdir(diretorio_base):
        caminho_empresa = os.path.join(diretorio_base, nome_empresa)

        # Pula se não for um diretório (ex: arquivos no nível raiz)
        if not os.path.isdir(caminho_empresa):
            continue

        # Lista para armazenar as mensagens de problemas de cada ano da empresa atual
        mensagens_problemas_empresa = []

        # Percorre as pastas de cada ano dentro da pasta da empresa
        for nome_pasta_ano in os.listdir(caminho_empresa):
            caminho_ano = os.path.join(caminho_empresa, nome_pasta_ano)

            # Pula se não for um diretório (ex: arquivos no nível da empresa)
            if not os.path.isdir(caminho_ano):
                continue

            # Flag para indicar se o ano atual tem algum problema
            ano_tem_problema = False
            # Lista temporária para as mensagens deste ano
            mensagens_ano_atual = []

            caminho_jsons = os.path.join(caminho_ano, "jsons")

            if os.path.exists(caminho_jsons) and os.path.isdir(caminho_jsons):
                arquivos_json = [f for f in os.listdir(caminho_jsons) if f.lower().endswith(".json")]
                total_jsons = len(arquivos_json)

                count_raw_text_valido = 0       # raw_text existe e tem >= 100 caracteres
                count_raw_text_vazio_ausente = 0 # raw_text é nulo, vazio ou só espaços
                count_raw_text_curto = 0         # raw_text existe, mas tem < 100 caracteres
                count_erros_leitura = 0

                if total_jsons > 0:
                    for nome_arquivo_json in arquivos_json:
                        caminho_completo_json = os.path.join(caminho_jsons, nome_arquivo_json)
                        try:
                            with open(caminho_completo_json, "r", encoding="utf-8") as f:
                                dados_json = json.load(f)

                            raw_text = dados_json.get("raw_text")

                            if not raw_text or not str(raw_text).strip():
                                count_raw_text_vazio_ausente += 1
                            else:
                                # Se não está vazio/ausente, verifica o comprimento
                                if len(raw_text) < 100:
                                    count_raw_text_curto += 1
                                else:
                                    count_raw_text_valido += 1
                        except Exception as e:
                            count_erros_leitura += 1
                            # Opcional: print(f"    [ERRO DETALHADO] {caminho_completo_json}: {e}")

                    # Verifica se houve *qualquer* problema para este ano
                    if count_raw_text_vazio_ausente > 0 or \
                       count_raw_text_curto > 0 or \
                       count_erros_leitura > 0:
                        ano_tem_problema = True
                        mensagens_ano_atual.append(f"  Ano {nome_pasta_ano}: {total_jsons} JSONs encontrados.")
                        # Sempre mostra o total válido, mesmo que seja 0, para contexto
                        mensagens_ano_atual.append(f"    - {count_raw_text_valido} com 'raw_text' válido (>= 100 chars).")
                        if count_raw_text_vazio_ausente > 0:
                            mensagens_ano_atual.append(f"    - **{count_raw_text_vazio_ausente}** com 'raw_text' vazio ou ausente.")
                        if count_raw_text_curto > 0:
                            mensagens_ano_atual.append(f"    - **{count_raw_text_curto}** com 'raw_text' muito curto (< 100 chars).")
                        if count_erros_leitura > 0:
                            mensagens_ano_atual.append(f"    - **{count_erros_leitura}** com erros de leitura/parsing.")
                else: # total_jsons == 0 (nenhum JSON encontrado na pasta 'jsons')
                    ano_tem_problema = True
                    mensagens_ano_atual.append(f"  Ano {nome_pasta_ano}: NENHUM JSON encontrado na pasta 'jsons'.")
            else: # Pasta 'jsons' não encontrada
                ano_tem_problema = True
                mensagens_ano_atual.append(f"  Ano {nome_pasta_ano}: Pasta 'jsons' **não encontrada** ou não é um diretório.")

            # Se este ano teve algum problema, adiciona suas mensagens à lista da empresa
            if ano_tem_problema:
                mensagens_problemas_empresa.extend(mensagens_ano_atual)

        # Após verificar todos os anos de uma empresa, se houver mensagens de problema, imprima
        if mensagens_problemas_empresa:
            print(f"\n### Empresa: {nome_empresa} (Problemas Encontrados) ###")
            for msg in mensagens_problemas_empresa:
                print(msg)

caminho_base_pdfs = r"E:\Projetos\Github_ViniciusJ\Pesquisa_IC_IA\crawler-relacao-investidor\pdfs"

verificar_jsons_e_raw_text_em_empresas_problemas_apenas(caminho_base_pdfs)


--- Verificando arquivos JSON e 'raw_text' na pasta base: E:\Projetos\Github_ViniciusJ\Pesquisa_IC_IA\crawler-relacao-investidor\pdfs (Apenas Problemas) ---



### Avaliação da saida de txt

In [12]:
# Caminho da pasta onde estão as subpastas MAGALU, Casas Bahia, e GPA
caminho_base = r"E:\Projetos\Github_ViniciusJ\Pesquisa_IC_IA\crawler-relacao-investidor\pdfs\Gerdau"

# Lista para armazenar os dados
dados_txt = []

# Percorre todas as pastas e subpastas
for raiz, diretorios, arquivos in os.walk(caminho_base):
    # Verifica se está dentro da pasta 'textos' para processar os arquivos
    if 'textos' in raiz:
        for nome_arquivo in arquivos:
            if nome_arquivo.endswith(".txt"):
                caminho_completo = os.path.join(raiz, nome_arquivo)
                try:
                    # Abre o arquivo .txt e calcula o número total de caracteres
                    with open(caminho_completo, "r", encoding="utf-8") as f:
                        texto = f.read()

                    # Armazena o nome do arquivo e a soma dos caracteres
                    dados_txt.append([nome_arquivo, len(texto)])

                except Exception as e:
                    dados_txt.append([nome_arquivo, f"Erro: {e}"])

# Cria o DataFrame a partir da lista
df_txt = pd.DataFrame(dados_txt, columns=["Arquivo", "Total_caracteres"])

# Exibe o DataFrame
print(df_txt)


           Arquivo  Total_caracteres
0   2015_1T_RR.txt             38301
1   2015_2T_RR.txt             39706
2   2015_3T_RR.txt             41840
3   2015_4T_RR.txt             44480
4   2015_1T_AP.txt              6037
..             ...               ...
69  2024_4T_AP.txt             17266
70  2025_1T_RR.txt             48711
71  2025_2T_RR.txt             57094
72  2025_1T_AP.txt             12679
73  2025_2T_AP.txt             14372

[74 rows x 2 columns]


In [13]:
vazios_str = df_txt[df_txt['Total_caracteres'] == 0]
print(vazios_str)

Empty DataFrame
Columns: [Arquivo, Total_caracteres]
Index: []


### Avaliação dos arquivos individualmente

In [32]:
caminho_pdf = "E:\\Projetos\\Github_ViniciusJ\\Pesquisa_IC_IA\\crawler-relacao-investidor\\pdfs\\Gerdau\\2015\\originais\\2015_2T_RR.pdf"

In [33]:
def diagnosticar_pdf(caminho_pdf):
    print(f"\nAnalisando: {caminho_pdf}")

    # 1. Verifica número de páginas com PyPDF2
    try:
        reader = PdfReader(caminho_pdf)
        num_paginas = len(reader.pages)
        print(f"Número de páginas: {num_paginas}")
    except Exception as e:
        print(f"Erro ao ler número de páginas: {e}")
        return

    # 2. Extrai texto com pdfminer
    try:
        texto = extract_text(caminho_pdf)
        total_caracteres = len(texto.strip())
        print(f"Caracteres extraídos: {total_caracteres}")
    except Exception as e:
        print(f"Erro ao extrair texto: {e}")
        return

In [34]:
diagnosticar_pdf(caminho_pdf)


Analisando: E:\Projetos\Github_ViniciusJ\Pesquisa_IC_IA\crawler-relacao-investidor\pdfs\Gerdau\2015\originais\2015_2T_RR.pdf
Número de páginas: 16
Caracteres extraídos: 49961


In [31]:
def visualizar_pdf(path_pdf, preview_chars=500):
    print(f"\n Analisando: {path_pdf}")
    
    try:
        reader = PdfReader(caminho_pdf)
        num_paginas = len(reader.pages)
        print(f"Número de páginas: {num_paginas}")
        texto = extract_text(path_pdf)
        total = len(texto)
        print(f"Total de caracteres extraídos: {total}")

        if total == 0:
            print(" Texto vazio!")
            return
        
        # Visualização: primeiras e últimas partes
        print("\n INÍCIO DO TEXTO:")
        print(texto[:preview_chars])

        print("\n FIM DO TEXTO:")
        print(texto[-preview_chars:])

    except Exception as e:
        print(f"Erro ao extrair texto: {e}")


In [35]:
visualizar_pdf(caminho_pdf)


 Analisando: E:\Projetos\Github_ViniciusJ\Pesquisa_IC_IA\crawler-relacao-investidor\pdfs\Gerdau\2015\originais\2015_2T_RR.pdf
Número de páginas: 16
Total de caracteres extraídos: 50287

 INÍCIO DO TEXTO:
Destaques do 2º trimestre de 2015 

Principais Destaques 

•  Manutenção dos níveis de EBITDA consolidado e de margem EBITDA, apesar da sobreoferta de aço 

mundial e das adversidades econômicas no Brasil.  

•  Maiores  volumes  e  melhor  metal  spread  da  Operação  de  Negócio  América  do  Norte  no  2T15 
parcialmente  compensaram  o  menor  desempenho  da  Operação  de  Negócio  Brasil,  quando 
comparado com o 1T15.  

•  Redução de 6,4% nas despesas com vendas, gerais e administrativas 

 FIM DO TEXTO:
no início do período
Caixa e equivalentes de caixa no final do período

             740.549 
          3.049.971 

3.790.520

             528.837 

2.099.224
2.628.061

 16 

 
 
 
 
 
           
           
            
           
           
               
        
     

In [36]:
from pdfminer.high_level import extract_text
import pdfplumber
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError
from pathlib import Path


def extrair_com_pdfminer(caminho_pdf: str) -> tuple[str, int]:
    """
    Extrai texto de um PDF usando pdfminer.
    Retorna (texto, numero_paginas).
    """
    try:
        texto = extract_text(caminho_pdf)
    except PDFSyntaxError as e:
        print(f"[pdfminer] Erro de sintaxe no PDF {caminho_pdf}: {e}")
        return "", 0
    except Exception as e:
        print(f"[pdfminer] Erro ao extrair texto de {caminho_pdf}: {e}")
        return "", 0

    # Descobrir o número de páginas (pdfminer não facilita isso).
    # Podemos usar pdfplumber só para contar:
    num_paginas = 0
    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            num_paginas = len(pdf.pages)
    except Exception:
        num_paginas = -1  # fallback se der erro

    return texto.strip(), num_paginas

In [37]:
import pdfplumber

def extrair_com_pdfplumber(caminho_pdf: str) -> tuple[str, int]:
    """
    Extrai texto de um PDF usando pdfplumber.
    Retorna (texto, numero_paginas).
    """
    texto = ""
    num_paginas = 0
    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            num_paginas = len(pdf.pages)
            for pagina in pdf.pages:
                pagina_texto = pagina.extract_text()
                if pagina_texto:
                    texto += pagina_texto + "\n"
        return texto.strip(), num_paginas
    except Exception as e:
        print(f"[pdfplumber] Erro ao extrair texto de {caminho_pdf}: {e}")
        return "", 0


In [41]:
caminho = r"E:\\Projetos\\Github_ViniciusJ\\Pesquisa_IC_IA\\crawler-relacao-investidor\\pdfs\\Gerdau\\2015\\originais\\2015_2T_RR.pdf"

texto_pdfminer, paginas_pdfminer = extrair_com_pdfminer(caminho)
texto_pdfplumber, paginas_pdfplumber = extrair_com_pdfplumber(caminho)

print(f"[pdfminer] Páginas: {paginas_pdfminer} | Texto: {len(texto_pdfminer)} caracteres")
print(f"[pdfplumber] Páginas: {paginas_pdfplumber} | Texto: {len(texto_pdfplumber)} caracteres")

[pdfminer] Páginas: 16 | Texto: 49961 caracteres
[pdfplumber] Páginas: 16 | Texto: 40340 caracteres
